In [1]:
import sys, os
from pathlib import Path


import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from env.drone_env        import DroneEnv2D
from env.discrete_wrapper import DiscreteWrapper


In [2]:
base = DroneEnv2D(world_size=20.0, max_steps=300, level=0)
env = DiscreteWrapper(base, grid_size=20)

print(f'States: {env.n_states}  Actions: {env.n_actions}')



States: 400  Actions: 25


## 1. Sanity check — random policy

> Establishes a baseline. Every algorithm below should beat this.


In [3]:
rewards = []
for _ in range(200):
    s, _ = env.reset()
    G = 0.0
    while True:
        a = env.action_space.sample()
        s, r, t1, t2, _ = env.step(a)
        G += r
        if t1 or t2:
            break
    rewards.append(G)

print(f'Random policy  mean reward: {np.mean(rewards):.2f}')


Random policy  mean reward: -10.44


## 2. Dynamic Programming — Policy Iteration & Value Iteration

Requires a model. We approximate P(s'|s,a) and R(s,a) by sampling
the discrete env exhaustively, then solve the Bellman equations exactly.

Note: this builds a model over 400 states × 9 actions, which takes
a couple of minutes. Skip this cell if you're short on time —
the other algorithms don't depend on it.

In [ ]:

# %%
from algos.classical.dynamic_programming import build_model, policy_iteration, value_iteration
from algos.classical.dynamic_programming import policy_improvement


P, R   = build_model(env, n_samples=3)

pi_policy, V_pi = policy_iteration(P, R, gamma=0.95, theta=1e-4)
vi_policy, V_vi = value_iteration(P, R, gamma=0.95, theta=1e-4)

print(f'\nV_pi range : [{V_pi.min():.2f}, {V_pi.max():.2f}]')
print(f'V_vi range : [{V_vi.min():.2f}, {V_vi.max():.2f}]')

Building environment model...


States:   0%|          | 0/400 [00:00<?, ?it/s]

States: 100%|██████████| 400/400 [00:19<00:00, 20.76it/s]



=== Policy Iteration ===
  Iteration 1
  Iteration 2
  Iteration 3
  Iteration 4
  Iteration 5
  Iteration 6
  Iteration 7
  Iteration 8
  Iteration 9
  Iteration 10
  Iteration 11
  Iteration 12
  Iteration 13
  Iteration 14
  Iteration 15
  Iteration 16
  Iteration 17
  Iteration 18
  Iteration 19
  Iteration 20
  Iteration 21
  Iteration 22
  Iteration 23
  Iteration 24
  Iteration 25
  Iteration 26
  Iteration 27
  Iteration 28
  Iteration 29
  Iteration 30
  Iteration 31
  Iteration 32
  Iteration 33
  Iteration 34
  Iteration 35
  Iteration 36
  Iteration 37
  Iteration 38
  Iteration 39
  Iteration 40
  Iteration 41
  Iteration 42
  Iteration 43
  Iteration 44
  Iteration 45
  Iteration 46
  Iteration 47
  Iteration 48
  Iteration 49
  Iteration 50
  Iteration 51
  Iteration 52
  Iteration 53
  Iteration 54
  Iteration 55
  Iteration 56
  Iteration 57
  Iteration 58
  Iteration 59
  Iteration 60
  Iteration 61
  Iteration 62
  Iteration 63
  Iteration 64
  Iteration 65
  Iterat

In [15]:
import pygame

base_vis = DroneEnv2D(world_size=20, level=1, render_mode="human", fps=8)
env_vis = DiscreteWrapper(base_vis, grid_size=20)

state, info = env_vis.reset(seed=42)
total_reward = 0
env_vis.render()

running = True
for step in range(500):
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
            break
    if not running:
        break
    
    action = int(np.argmax(pi_policy[state]))     # <-- replaces env.action_space.sample()
    state, reward, terminated, truncated, info = env_vis.step(action)
    total_reward += reward
    env_vis.render()
    if terminated or truncated:
        print(f"done at step {step}, reward={total_reward:.2f}")
        break

env_vis.close()

done at step 16, reward=-3.06


In [ ]:

# # %% [markdown]
# # ## 1. Sanity check — random policy

# # Establishes a baseline. Every algorithm below should beat this.

# # %%
# rewards = []
# for _ in range(200):
#     s, _ = env.reset()
#     G = 0.0
#     while True:
#         a = env.action_space.sample()
#         s, r, t1, t2, _ = env.step(a)
#         G += r
#         if t1 or t2:
#             break
#     rewards.append(G)

# print(f'Random policy  mean reward: {np.mean(rewards):.2f}')


# # %% [markdown]
# # ## 2. Q-Learning
# #
# # Off-policy TD control.
# # δ = r + γ max_a' Q(s',a') - Q(s,a)

# # %%
# from algos.classical import QLearning

# ql = QLearning(env.n_states, env.n_actions, gamma=0.99, alpha=0.1)
# ql.train(make_env(), n_episodes=3000, log_every=300)


# # %% [markdown]
# # ## 3. SARSA — compare with Q-Learning
# #
# # On-policy TD control.
# # δ = r + γ Q(s',a') - Q(s,a)   where a' is sampled from ε-greedy

# # %%
# from algos.classical import SARSA

# sarsa = SARSA(env.n_states, env.n_actions, gamma=0.99, alpha=0.1)
# sarsa.train(make_env(), n_episodes=3000, log_every=300)


# # %% [markdown]
# # ## 4. Monte Carlo
# #
# # Model-free, learns from complete episodes.
# # Q(s,a) ← Q(s,a) + α [G_t - Q(s,a)]

# # %%
# from algos.classical import MonteCarloControl

# mc = MonteCarloControl(env.n_states, env.n_actions,
#                        gamma=0.99, alpha=0.05, first_visit=True)
# mc.train(make_env(), n_episodes=2000, log_every=200)


# # %% [markdown]
# # ## 5. Dynamic Programming — Policy Iteration & Value Iteration
# #
# # Requires a model. We approximate P(s'|s,a) and R(s,a) by sampling
# # the discrete env exhaustively, then solve the Bellman equations exactly.
# #
# # Note: this builds a model over 400 states × 9 actions, which takes
# # a couple of minutes. Skip this cell if you're short on time —
# # the other algorithms don't depend on it.

# # %%
# from algos.classical import build_model, policy_iteration, value_iteration
# from algos.classical.dynamic_programming import policy_improvement

# dp_env = make_env(level=0)
# P, R   = build_model(dp_env, n_samples=3)

# pi_policy, V_pi = policy_iteration(P, R, gamma=0.99, theta=1e-4)
# vi_policy, V_vi = value_iteration(P, R, gamma=0.99, theta=1e-4)

# print(f'\nV_pi range : [{V_pi.min():.2f}, {V_pi.max():.2f}]')
# print(f'V_vi range : [{V_vi.min():.2f}, {V_vi.max():.2f}]')


# # %% [markdown]
# # ## 6. Learning curves — all online algorithms
# #
# # Things to look for:
# # - **Q-Learning** often rises fastest early — more aggressive off-policy learning.
# # - **SARSA** may be slightly lower but more stable — it accounts for its own exploration.
# # - **Monte Carlo** starts slower — needs full episodes before any update happens.

# # %%
# def smooth(x, w=50):
#     return np.convolve(x, np.ones(w) / w, mode='valid')

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# for agent, label in [(ql, 'Q-Learning'), (sarsa, 'SARSA'), (mc, 'Monte Carlo')]:
#     ax1.plot(smooth(agent.episode_rewards), label=label)
#     ax2.plot(smooth(agent.episode_lengths), label=label)

# for ax, title, ylabel in [(ax1, 'Episode Reward', 'reward'),
#                            (ax2, 'Episode Length', 'steps')]:
#     ax.set_title(title)
#     ax.set_xlabel('episode')
#     ax.set_ylabel(ylabel)
#     ax.legend()
#     ax.grid(alpha=0.3)

# plt.tight_layout()
# plt.show()


# # %% [markdown]
# # ## 7. Visualise the Q-table as a heatmap + policy arrows
# #
# # V(s) = max_a Q(s,a)  — shows how "good" each grid cell is.
# # π*(s) = argmax_a Q(s,a)  — shows which direction the agent thrusts from each cell.

# # %%
# V  = np.max(ql.Q, axis=1).reshape(env.grid_size, env.grid_size)
# PI = np.argmax(ql.Q, axis=1).reshape(env.grid_size, env.grid_size)

# # Action index -> (dx, dy) direction, matching DiscreteWrapper.ACTION_VECTORS
# DX = [ 0, 1, 1, 1, 0, -1, -1, -1, 0]
# DY = [-1, -1, 0, 1, 1,  1,  0, -1, 0]

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# # Value heatmap
# im = axes[0].imshow(V, origin='upper', cmap='viridis')
# plt.colorbar(im, ax=axes[0])
# axes[0].set_title('V(s) = max_a Q(s,a)  —  Q-Learning')

# # Policy arrows over a faded value map
# axes[1].imshow(V, origin='upper', cmap='viridis', alpha=0.4)
# g = env.grid_size
# for row in range(g):
#     for col in range(g):
#         a = PI[row, col]
#         if a < 8:   # skip hover (action 8 has zero direction)
#             axes[1].annotate(
#                 '', xy=(col + 0.5 * DX[a], row + 0.5 * DY[a]),
#                 xytext=(col, row),
#                 arrowprops=dict(arrowstyle='->', color='white', lw=0.8),
#             )
# axes[1].set_title('Greedy policy  π*(s) = argmax_a Q(s,a)')

# plt.tight_layout()
# plt.show()


# # %% [markdown]
# # ## 8. Key observations
# #
# # - The **V(s) heatmap** should show high values near where the goal tends
# #   to spawn, and low values near obstacles/borders.
# # - The **policy arrows** should generally point toward higher-value regions.
# # - All three online algorithms should comfortably beat the random baseline
# #   from section 1.
# #
# # The plateau you'll observe here is the ceiling of tabular RL: velocity,
# # obstacle rays, and fine-grained position are all invisible to the agent
# # (we only kept a 20x20 grid over x,y). That's the motivation for Phase 2 —
# # deep RL, where a neural network can consume the full 14-dim continuous state.